<div dir="rtl" style="text-align:right; line-height:1.9;">
  <h2>01 — ربط Google Drive</h2>

  <p>
    في هذه الخلية نقوم بربط بيئة
    <span dir="ltr">Google Colab</span>
    مع
    <span dir="ltr">Google Drive</span>
    حتى نستطيع قراءة ملفات المشروع وحفظ نواتج تعدين الأزواج داخل مجلد المشروع.
  </p>

  <p>
    هذه الخطوة ضرورية لأن ملفات البيانات والنماذج والـ
    <span dir="ltr">embeddings</span>
    محفوظة داخل
    <span dir="ltr">Drive</span>
    وليست داخل التخزين المؤقت الخاص بـ
    <span dir="ltr">Colab</span>.
  </p>
</div>

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<div dir="rtl" style="text-align:right; line-height:1.9;">
  <h2>02 — تحديد مسارات المشروع وملفات التدريب</h2>

  <p>
    في هذه الخلية نحدد المسار الأساسي للمشروع داخل
    <span dir="ltr">Google Drive</span>
    ثم نعرض الملفات الموجودة داخل مجلد
    <span dir="ltr">training_data</span>.
  </p>

  <p>
    الهدف من هذه الخطوة هو التأكد من وجود ملفات التقسيم، التنظيف، وملفات البيانات التي سنستخدمها في بناء أزواج التدريب.
  </p>
</div>

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")
TRAINING_DATA_DIR = PROJECT_ROOT / "training_data"

for p in TRAINING_DATA_DIR.rglob("*"):
    if p.is_file():
        print(p)

/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/post_aware_pairs_20260420_172535/training_pairs_v2.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/post_aware_pairs_20260420_172535/training_pairs_v2_metadata.json
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/post_aware_pairs_20260420_172535/training_pairs_v2_preview.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/post_aware_pairs_v3_20260420_174426/training_pairs_v3.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/post_aware_pairs_v3_20260420_174426/clean_comments_preview.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/post_aware_pairs_v3_20260420_174426/pairs_preview.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/post_aware_pairs_v3_20260420_174426/training_pairs_v3_metadata.json
/content/drive

<div dir="rtl" style="text-align:right; line-height:1.9;">
  <h2>03 — استيراد المكتبات وتحديد المسارات الأساسية</h2>

  <p>
    في هذه الخلية نستورد المكتبات المستخدمة في قراءة البيانات، معالجة الجداول، التعامل مع المسارات، والتحكم بالعشوائية.
  </p>

  <p>
    كما يتم تحديد المسارات الأساسية لملفات التعليقات الخام، المنشورات الخام، وملفات
    <span dir="ltr">post-level split</span>
    التي سنستخدمها لاحقًا لربط التعليقات بأقسام التدريب والتحقق.
  </p>
</div>

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import random
import time
from IPython.display import display

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

RAW_COMMENTS_PATH = PROJECT_ROOT / "fb_news_comments_1000K_hashed.csv"
RAW_POSTS_PATH = PROJECT_ROOT / "fb_news_posts_20K.csv"

SPLIT_DIR = PROJECT_ROOT / "training_data/v5_post_level_split_20260420_193523"
POST_SPLIT_PATH = SPLIT_DIR / "post_split_manifest.csv"
COMMENT_SPLIT_PATH = SPLIT_DIR / "comment_split_manifest_minimal.csv"

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / f"training_data/v7_raw_pair_mining_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_COMMENTS_PATH:", RAW_COMMENTS_PATH.exists(), RAW_COMMENTS_PATH)
print("RAW_POSTS_PATH:", RAW_POSTS_PATH.exists(), RAW_POSTS_PATH)
print("POST_SPLIT_PATH:", POST_SPLIT_PATH.exists(), POST_SPLIT_PATH)
print("COMMENT_SPLIT_PATH:", COMMENT_SPLIT_PATH.exists(), COMMENT_SPLIT_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

RAW_COMMENTS_PATH: True /content/drive/MyDrive/semester project/news_comment_topic_system/fb_news_comments_1000K_hashed.csv
RAW_POSTS_PATH: True /content/drive/MyDrive/semester project/news_comment_topic_system/fb_news_posts_20K.csv
POST_SPLIT_PATH: True /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v5_post_level_split_20260420_193523/post_split_manifest.csv
COMMENT_SPLIT_PATH: True /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v5_post_level_split_20260420_193523/comment_split_manifest_minimal.csv
OUTPUT_DIR: /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_pair_mining_20260428_230310


<div dir="rtl" style="text-align:right; line-height:1.9;">
  <h2>04 — فحص أولي لبنية الملفات الخام وملفات التقسيم</h2>

  <p>
    في هذه الخلية نقوم بقراءة عينة صغيرة فقط من ملفات البيانات الخام باستخدام
    <span dir="ltr">nrows=5</span>.
    الهدف هنا ليس تحميل البيانات كاملة، بل فحص أسماء الأعمدة وشكل الملفات قبل بدء المعالجة الأساسية.
  </p>

  <p>
    يتم فحص ملف التعليقات الخام
    <span dir="ltr">RAW_COMMENTS_PATH</span>
    وملف المنشورات الخام
    <span dir="ltr">RAW_POSTS_PATH</span>
    للتأكد من وجود الأعمدة الأساسية مثل
    <span dir="ltr">created_time</span>،
    <span dir="ltr">message</span>،
    و
    <span dir="ltr">post_name</span>.
  </p>

  <p>
    كما يتم فحص وجود ملفات التقسيم السابقة مثل
    <span dir="ltr">POST_SPLIT_PATH</span>
    و
    <span dir="ltr">COMMENT_SPLIT_PATH</span>
    لأن عملية تعدين الأزواج يجب أن تعتمد على تقسيم
    <span dir="ltr">post-level split</span>
    لتجنب
    <span dir="ltr">data leakage</span>.
  </p>

  <p>
    <strong>إشارة التأكد:</strong>
    أنت في الخلية الصحيحة إذا كانت الخلية تستخدم
    <span dir="ltr">pd.read_csv(..., nrows=5)</span>
    وتعرض أسماء أعمدة
    <span dir="ltr">raw comments</span>
    و
    <span dir="ltr">raw posts</span>.
  </p>
</div>

In [ ]:
comments_df = pd.read_csv(RAW_COMMENTS_PATH, nrows=5)
posts_df = pd.read_csv(RAW_POSTS_PATH, nrows=5)

print("Raw comments columns:")
print(comments_df.columns.tolist())
display(comments_df)

print("\nRaw posts columns:")




print(posts_df.columns.tolist())
display(posts_df)

if POST_SPLIT_PATH.exists():
    post_split_preview = pd.read_csv(POST_SPLIT_PATH, nrows=5)
    print("\nPost split columns:")
    print(post_split_preview.columns.tolist())
    display(post_split_preview)

if COMMENT_SPLIT_PATH.exists():
    comment_split_preview = pd.read_csv(COMMENT_SPLIT_PATH, nrows=5)
    print("\nComment split columns:")
    print(comment_split_preview.columns.tolist())
    display(comment_split_preview)

Raw comments columns:
['created_time', 'from_id', 'from_name', 'message', 'post_name']


,created_time,from_id,from_name,message,post_name
0,2017-07-14T14:43:54+0000,33661642d99eeceeb086,4ca212f16b9f954d5e0a,We are speaking to NRA supporters as well as W...,33661642d99eeceeb086_10154890879532217
1,2017-07-14T14:41:59+0000,33661642d99eeceeb086,4ca212f16b9f954d5e0a,If you are just joining us we are outside of t...,33661642d99eeceeb086_10154890879532217
2,2017-07-14T14:41:58+0000,142b054b9d7f119260fa,210f51cc65568bf2d528,Do you know how backward America are in allowi...,142b054b9d7f119260fa_10154890879532217
3,2017-07-14T14:42:25+0000,52228acb5d5ca468be8c,46377b3cde64b2bf93dc,People who legally own guns often seem all too...,52228acb5d5ca468be8c_10154890879532217
4,2017-07-14T14:40:46+0000,af7fe02906a110370810,087accbb0dc7c975d194,Have you snowflakes watched the news and seen ...,af7fe02906a110370810_10154890879532217



Raw posts columns:
['created_time', 'description', 'link', 'message', 'page_id', 'post_id', 'react_angry', 'react_haha', 'react_like', 'react_love', 'react_sad', 'react_wow', 'scrape_time', 'shares']


,created_time,description,link,message,page_id,post_id,react_angry,react_haha,react_like,react_love,react_sad,react_wow,scrape_time,shares
0,2017-07-14T14:30:59+0000,NaN,https://www.facebook.com/bbcnews/videos/101548...,We are #LIVE outside the National Rifle Associ...,228735667216,228735667216_10154890879532217,54,24,993,144,12,24,2017-07-14 11:01:24.379857,139
1,2017-07-14T14:20:59+0000,,http://bbc.in/2talMsx,UPDATE: \r\n-2 Ukrainian tourists killed in st...,228735667216,228735667216_10154890968202217,172,8,994,11,783,264,2017-07-14 11:01:24.379857,680
2,2017-07-14T13:40:38+0000,NaN,https://www.facebook.com/bbcnews/videos/101548...,Proms: Come with us on a tour of the Royal Alb...,228735667216,228735667216_10154890852247217,5,12,2034,369,6,45,2017-07-14 11:01:24.379857,395
3,2017-07-14T12:55:45+0000,NaN,https://www.facebook.com/bbcnews/videos/142678...,Thousands say their final goodbyes to Bradley ...,228735667216,228735667216_1426789250735491,6,0,2262,754,1989,11,2017-07-14 11:01:24.379857,542
4,2017-07-14T12:45:00+0000,NaN,https://www.facebook.com/bbcnews/videos/101548...,"Despite safety warnings, this beach near an ai...",228735667216,228735667216_10154890645702217,65,513,4336,54,128,815,2017-07-14 11:01:24.379857,1956



Post split columns:
['post_id_true', 'comment_count', 'page_source', 'split']


,post_id_true,comment_count,page_source,split
0,1387642564627948,27,1.019886e+14,train
1,1218169058241967,82,1.019886e+14,train
2,1338491556209716,66,1.019886e+14,train
3,1293490997376439,100,1.019886e+14,train
4,1353297751395763,57,1.019886e+14,train



Comment split columns:
['post_id_true', 'split']


,post_id_true,split
0,10154890879532217,train
1,10154890879532217,train
2,10154890879532217,train
3,10154890879532217,train
4,10154890879532217,train


<div dir="rtl" style="text-align:right; line-height:1.9;">
  <h2>05 — تحميل التعليقات الخام وتقسيم المنشورات</h2>

  <p>
    في هذه الخلية نقوم بتحميل ملف التعليقات الخام كاملًا، إضافة إلى ملف
    <span dir="ltr">post-level split</span>
    الذي يحدد القسم الخاص بكل منشور.
  </p>

  <p>
    يحتوي ملف التعليقات الخام على التعليقات الأصلية وحقولها الأساسية، مثل:
    <span dir="ltr">created_time</span>،
    <span dir="ltr">from_id</span>،
    <span dir="ltr">from_name</span>،
    <span dir="ltr">message</span>،
    و
    <span dir="ltr">post_name</span>.
  </p>

  <p>
    أما ملف
    <span dir="ltr">post-level split</span>
    فيحدد ما إذا كان المنشور ينتمي إلى
    <span dir="ltr">train</span>
    أو
    <span dir="ltr">validation</span>
    أو
    <span dir="ltr">test</span>.
  </p>

  <p>
    هذه الخطوة مهمة لأننا لن نبني الأزواج عشوائيًا من كل البيانات، بل سنحافظ على تقسيم المنشورات بحيث تبقى جميع تعليقات المنشور الواحد داخل القسم نفسه.
    هذا يمنع تسرب البيانات
    <span dir="ltr">data leakage</span>
    بين التدريب والتحقق والاختبار.
  </p>

  <p>
    <strong>إشارة التأكد:</strong>
    أنت في الخلية الصحيحة إذا كانت الخلية تحمل
    <span dir="ltr">comments</span>
    من
    <span dir="ltr">RAW_COMMENTS_PATH</span>
    وتحمل
    <span dir="ltr">post_split</span>
    من
    <span dir="ltr">POST_SPLIT_PATH</span>
    وتطبع
    <span dir="ltr">Raw comments shape</span>
    و
    <span dir="ltr">Post split shape</span>.
  </p>
</div>

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

# Load raw comments
comments = pd.read_csv(RAW_COMMENTS_PATH)

# Load post-level split
post_split = pd.read_csv(POST_SPLIT_PATH)

print("Raw comments shape:", comments.shape)
print("Post split shape:", post_split.shape)

print("\nRaw comments columns:")
print(comments.columns.tolist())

print("\nPost split columns:")
print(post_split.columns.tolist())

display(comments.head())
display(post_split.head())

Raw comments shape: (1038319, 5)
Post split shape: (17351, 4)

Raw comments columns:
['created_time', 'from_id', 'from_name', 'message', 'post_name']

Post split columns:
['post_id_true', 'comment_count', 'page_source', 'split']


,created_time,from_id,from_name,message,post_name
0,2017-07-14T14:43:54+0000,33661642d99eeceeb086,4ca212f16b9f954d5e0a,We are speaking to NRA supporters as well as W...,33661642d99eeceeb086_10154890879532217
1,2017-07-14T14:41:59+0000,33661642d99eeceeb086,4ca212f16b9f954d5e0a,If you are just joining us we are outside of t...,33661642d99eeceeb086_10154890879532217
2,2017-07-14T14:41:58+0000,142b054b9d7f119260fa,210f51cc65568bf2d528,Do you know how backward America are in allowi...,142b054b9d7f119260fa_10154890879532217
3,2017-07-14T14:42:25+0000,52228acb5d5ca468be8c,46377b3cde64b2bf93dc,People who legally own guns often seem all too...,52228acb5d5ca468be8c_10154890879532217
4,2017-07-14T14:40:46+0000,af7fe02906a110370810,087accbb0dc7c975d194,Have you snowflakes watched the news and seen ...,af7fe02906a110370810_10154890879532217


,post_id_true,comment_count,page_source,split
0,1387642564627948,27,101988643193353.0,train
1,1218169058241967,82,101988643193353.0,train
2,1338491556209716,66,101988643193353.0,train
3,1293490997376439,100,101988643193353.0,train
4,1353297751395763,57,101988643193353.0,train


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>06 — استخراج وتوحيد معرف المنشور</h2>

  <p>
    تستخرج هذه الخلية معرف المنشور الحقيقي من عمود
    <span dir="ltr">post_name</span>
    داخل التعليقات، ثم توحّد صيغة المعرفات في التعليقات وملف التقسيم.
  </p>

  <p>
    الهدف هو تجهيز مفتاح ربط موحّد
    <span dir="ltr">post_id_true</span>
    لاستخدامه لاحقًا في ربط كل تعليق بالقسم الصحيح:
    <span dir="ltr">train</span> أو
    <span dir="ltr">val</span> أو
    <span dir="ltr">test</span>.
  </p>
</div>

In [ ]:
def extract_post_id_true(post_name):
    if pd.isna(post_name):
        return None

    post_name = str(post_name).strip()

    if "_" not in post_name:
        return None

    return post_name.split("_")[-1].strip()


def normalize_post_id(x):
    """
    يحوّل post_id إلى string نظيف.
    يفيد إذا قرأ pandas بعض الأرقام كـ float مثل 12345.0
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x


comments["post_id_true"] = comments["post_name"].apply(extract_post_id_true)
comments["post_id_true"] = comments["post_id_true"].apply(normalize_post_id)

post_split["post_id_true"] = post_split["post_id_true"].apply(normalize_post_id)

print("Raw comments:", len(comments))
print("Comments with extracted post_id_true:", comments["post_id_true"].notna().sum())
print("Unique comment post_id_true:", comments["post_id_true"].nunique())
print("Unique split post_id_true:", post_split["post_id_true"].nunique())

display(comments[["post_name", "post_id_true", "message"]].head(10))

Raw comments: 1038319
Comments with extracted post_id_true: 1038319
Unique comment post_id_true: 18830
Unique split post_id_true: 17351


,post_name,post_id_true,message
0,33661642d99eeceeb086_10154890879532217,10154890879532217,We are speaking to NRA supporters as well as W...
1,33661642d99eeceeb086_10154890879532217,10154890879532217,If you are just joining us we are outside of t...
2,142b054b9d7f119260fa_10154890879532217,10154890879532217,Do you know how backward America are in allowi...
3,52228acb5d5ca468be8c_10154890879532217,10154890879532217,People who legally own guns often seem all too...
4,af7fe02906a110370810_10154890879532217,10154890879532217,Have you snowflakes watched the news and seen ...
5,9a90817f183ebc174742_10154890879532217,10154890879532217,I don't understand why America wants to carry ...
6,5969e4a86e559f513d79_10154890879532217,10154890879532217,It is my right to legally protect my life as b...
7,858bf285623eeccb82f2_10154890879532217,10154890879532217,"I'm a gun owner, but the NRA are just terroris..."
8,858bf285623eeccb82f2_10154890879532217,10154890879532217,2nd amendment makes you think you are free fro...
9,334ea0160501ea43df96_10154890879532217,10154890879532217,It should be a legal requirement that everyone...


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>07 — ربط التعليقات بتقسيم المنشورات</h2>

  <p>
    تربط هذه الخلية التعليقات بملف
    <span dir="ltr">post-level split</span>
    باستخدام العمود الموحد
    <span dir="ltr">post_id_true</span>.
  </p>

  <p>
    بعد الربط، يحصل كل تعليق على قيمة
    <span dir="ltr">split</span>
    التي تحدد هل ينتمي إلى
    <span dir="ltr">train</span> أو
    <span dir="ltr">val</span> أو
    <span dir="ltr">test</span>.
  </p>

  <p>
    هذه الخطوة تمنع
    <span dir="ltr">data leakage</span>
    لأن جميع تعليقات المنشور الواحد تبقى داخل القسم نفسه.
  </p>
</div>

In [ ]:
post_split_unique = (
    post_split[["post_id_true", "split"]]
    .drop_duplicates(subset=["post_id_true"])
    .copy()
)

comments_with_split = comments.merge(
    post_split_unique,
    on="post_id_true",
    how="inner"
)

print("Raw comments:", len(comments))
print("Comments after post-level split join:", len(comments_with_split))

print("\nSplit distribution:")
display(comments_with_split["split"].value_counts())

print("\nUnique posts per split:")
display(
    comments_with_split
    .groupby("split")["post_id_true"]
    .nunique()
    .reset_index(name="unique_posts")
)

display(comments_with_split[["post_id_true", "split", "message"]].head(10))

Raw comments: 1038319
Comments after post-level split join: 1035495

Split distribution:


,count
split,
train,727751
val,154483
test,153261



Unique posts per split:


,split,unique_posts
0,test,2594
1,train,12147
2,val,2610


,post_id_true,split,message
0,10154890879532217,train,We are speaking to NRA supporters as well as W...
1,10154890879532217,train,If you are just joining us we are outside of t...
2,10154890879532217,train,Do you know how backward America are in allowi...
3,10154890879532217,train,People who legally own guns often seem all too...
4,10154890879532217,train,Have you snowflakes watched the news and seen ...
5,10154890879532217,train,I don't understand why America wants to carry ...
6,10154890879532217,train,It is my right to legally protect my life as b...
7,10154890879532217,train,"I'm a gun owner, but the NRA are just terroris..."
8,10154890879532217,train,2nd amendment makes you think you are free fro...
9,10154890879532217,train,It should be a legal requirement that everyone...


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>08 — تنظيف نصوص التعليقات وحساب الإحصاءات الأساسية</h2>

  <p>
    تنظف هذه الخلية نصوص التعليقات في العمود
    <span dir="ltr">message</span>
    بإزالة الروابط ووسوم
    <span dir="ltr">HTML</span>
    وتوحيد المسافات.
  </p>

  <p>
    تنشئ العمود
    <span dir="ltr">clean_text_v7</span>
    ثم تحسب
    <span dir="ltr">word_count</span>
    و
    <span dir="ltr">char_count</span>
    لقياس طول التعليقات بعد التنظيف.
  </p>

  <p>
    تعرض الخلية إحصاءات عدد الكلمات وعينة مقارنة بين النص الأصلي والمنظف للتحقق من جودة التنظيف.
  </p>
</div>

In [ ]:
import re
import html

URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_TAG_RE = re.compile(r"<.*?>")
WHITESPACE_RE = re.compile(r"\s+")

def clean_comment_text_v7(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    # Remove URLs only
    text = URL_RE.sub(" ", text)

    # Remove HTML tags if any
    text = HTML_TAG_RE.sub(" ", text)

    # Normalize whitespace
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = WHITESPACE_RE.sub(" ", text)

    return text.strip()


comments_with_split["clean_text_v7"] = comments_with_split["message"].apply(clean_comment_text_v7)

comments_with_split["word_count"] = comments_with_split["clean_text_v7"].apply(lambda x: len(x.split()))
comments_with_split["char_count"] = comments_with_split["clean_text_v7"].apply(len)

print("Cleaned comments:", len(comments_with_split))

print("\nWord count stats:")
display(comments_with_split["word_count"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

display(
    comments_with_split[
        ["message", "clean_text_v7", "word_count", "split"]
    ].head(10)
)

Cleaned comments: 1035495

Word count stats:


,word_count
count,1.035495e+06
mean,2.957718e+01
std,5.323044e+01
min,0.000000e+00
50%,1.700000e+01
75%,3.500000e+01
90%,6.000000e+01
95%,8.600000e+01
99%,2.090000e+02
max,2.049000e+03


,message,clean_text_v7,word_count,split
0,We are speaking to NRA supporters as well as W...,We are speaking to NRA supporters as well as W...,12,train
1,If you are just joining us we are outside of t...,If you are just joining us we are outside of t...,46,train
2,Do you know how backward America are in allowi...,Do you know how backward America are in allowi...,29,train
3,People who legally own guns often seem all too...,People who legally own guns often seem all too...,33,train
4,Have you snowflakes watched the news and seen ...,Have you snowflakes watched the news and seen ...,23,train
5,I don't understand why America wants to carry ...,I don't understand why America wants to carry ...,32,train
6,It is my right to legally protect my life as b...,It is my right to legally protect my life as b...,38,train
7,"I'm a gun owner, but the NRA are just terroris...","I'm a gun owner, but the NRA are just terroris...",13,train
8,2nd amendment makes you think you are free fro...,2nd amendment makes you think you are free fro...,40,train
9,It should be a legal requirement that everyone...,It should be a legal requirement that everyone...,36,train


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>09 — تصفية التعليقات منخفضة المعلومات</h2>

  <p>
    تحدد هذه الخلية التعليقات القصيرة أو الضعيفة دلاليًا قبل مرحلة
    <span dir="ltr">pair mining</span>
    باستخدام حدّ أدنى لعدد الكلمات والمحارف الأبجدية.
  </p>

  <p>
    تنشئ العمود
    <span dir="ltr">is_low_information</span>
    ثم تبني
    <span dir="ltr">filtered_comments</span>
    الذي يحتوي فقط على التعليقات الصالحة للتدريب.
  </p>

  <p>
    تعرض الخلية عدد التعليقات قبل وبعد التصفية، وتوزيع
    <span dir="ltr">split</span>
    مع أمثلة من التعليقات المحذوفة والمحتفظ بها للتحقق.
  </p>
</div>

In [ ]:
def alpha_char_count(text):
    if not isinstance(text, str):
        return 0
    return sum(ch.isalpha() for ch in text)


def looks_like_low_information(text, min_words=6, min_alpha_chars=15):
    if not isinstance(text, str):
        return True

    text = text.strip()
    words = text.split()

    if len(words) < min_words:
        return True

    if alpha_char_count(text) < min_alpha_chars:
        return True

    return False


MIN_WORDS_FOR_PAIR_MINING = 6
MIN_ALPHA_CHARS = 15

comments_with_split["is_low_information"] = comments_with_split["clean_text_v7"].apply(
    lambda x: looks_like_low_information(
        x,
        min_words=MIN_WORDS_FOR_PAIR_MINING,
        min_alpha_chars=MIN_ALPHA_CHARS
    )
)

filtered_comments = comments_with_split[
    (~comments_with_split["is_low_information"]) &
    (comments_with_split["clean_text_v7"].str.len() > 2)
].copy()

filtered_comments = filtered_comments.reset_index(drop=True)

print("Before filtering:", len(comments_with_split))
print("After filtering:", len(filtered_comments))
print("Removed:", len(comments_with_split) - len(filtered_comments))

print("\nSplit distribution after filtering:")
display(filtered_comments["split"].value_counts())

print("\nRemoved examples:")
display(
    comments_with_split[comments_with_split["is_low_information"]]
    [["clean_text_v7", "word_count", "split"]]
    .sample(min(20, comments_with_split["is_low_information"].sum()), random_state=RANDOM_STATE)
)

print("\nKept examples:")
display(
    filtered_comments[["clean_text_v7", "word_count", "split"]]
    .sample(min(20, len(filtered_comments)), random_state=RANDOM_STATE)
)

Before filtering: 1035495
After filtering: 843636
Removed: 191859

Split distribution after filtering:


,count
split,
train,593543
val,125614
test,124479



Removed examples:


,clean_text_v7,word_count,split
695796,Not surprising for average Americans....,5,train
196053,Philipp Saar,2,train
134012,Christopher Taylor,2,train
263879,His ignorance is showing.,4,train
216072,Charlie Chant,2,test
558520,No problem,2,train
957778,,0,train
297021,Thaer Mutlaq,2,train
183746,Go Venus!!,2,train
427162,Please just stop,3,train



Kept examples:


,clean_text_v7,word_count,split
28573,hoping for a speedy recovery! i grew up hearin...,11,train
452953,Looks like trump's inability to close his mout...,116,train
823917,He's going to be as big on the world stage as ...,16,train
594520,"The vast majority of trump ""supporters"" who po...",20,val
767709,"In the Conservative lexicon, ""Incentive"" = Gov...",26,test
835030,The fib and congress is spending a lot of mone...,33,train
619680,"""Congress has the ability to censure a sitting...",29,test
431775,Why is he meeting with anybody?,6,train
429323,But her first contact email states She represe...,103,train
247111,That's our wonderful Representatives . Do you ...,11,train


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>10 — تجهيز بيانات التدريب والتحقق لبناء الأزواج</h2>

  <p>
    تقسم هذه الخلية
    <span dir="ltr">filtered_comments</span>
    إلى
    <span dir="ltr">train_df</span>
    و
    <span dir="ltr">val_df</span>
    لاستخدامهما في مرحلة
    <span dir="ltr">pair mining</span>.
  </p>

  <p>
    تعرض عدد التعليقات وعدد المنشورات الفريدة في كل قسم، ثم تعرض عينات من النصوص المنظفة للتحقق من جاهزية البيانات.
  </p>
</div>

In [ ]:
train_df = filtered_comments[filtered_comments["split"] == "train"].copy().reset_index(drop=True)
val_df = filtered_comments[filtered_comments["split"] == "val"].copy().reset_index(drop=True)

print("Train comments for pair mining:", len(train_df))
print("Val comments for pair mining:", len(val_df))

print("\nTrain unique posts:", train_df["post_id_true"].nunique())
print("Val unique posts:", val_df["post_id_true"].nunique())

display(train_df[["post_id_true", "clean_text_v7", "word_count"]].head(10))
display(val_df[["post_id_true", "clean_text_v7", "word_count"]].head(10))

Train comments for pair mining: 593543
Val comments for pair mining: 125614

Train unique posts: 12023
Val unique posts: 2585


,post_id_true,clean_text_v7,word_count
0,10154890879532217,We are speaking to NRA supporters as well as W...,12
1,10154890879532217,If you are just joining us we are outside of t...,46
2,10154890879532217,Do you know how backward America are in allowi...,29
3,10154890879532217,People who legally own guns often seem all too...,33
4,10154890879532217,Have you snowflakes watched the news and seen ...,23
5,10154890879532217,I don't understand why America wants to carry ...,32
6,10154890879532217,It is my right to legally protect my life as b...,38
7,10154890879532217,"I'm a gun owner, but the NRA are just terroris...",13
8,10154890879532217,2nd amendment makes you think you are free fro...,40
9,10154890879532217,It should be a legal requirement that everyone...,36


,post_id_true,clean_text_v7,word_count
0,10154890399087217,Its easy to tell drug dealers as they all hang...,15
1,10154890399087217,Are they really? Or is this one of those thing...,55
2,10154890399087217,This post was obviously made by suburban mothe...,17
3,10154890399087217,So is the emoji movie just one giant drug deal...,11
4,10154890399087217,"Hey guys, I want to do this on the down-low, s...",25
5,10154890399087217,Literally everybody knows of at least one drug...,25
6,10154890399087217,Drug prohibition; solves nothing at all and cr...,30
7,10154890399087217,We are losing so many young people to the drug...,32
8,10154890399087217,"absolutely disgusted the way we use apps, ban ...",26
9,10154890399087217,ahh so we will all have to lose more of our pr...,38


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>11 — حساب عدد الأزواج الممكنة داخل كل منشور</h2>

  <p>
    تحسب هذه الخلية عدد التعليقات داخل كل منشور في
    <span dir="ltr">train_df</span>
    و
    <span dir="ltr">val_df</span>
    ثم تقدّر عدد أزواج
    <span dir="ltr">same-post pairs</span>
    الممكنة لكل منشور.
  </p>

  <p>
    تعرض إجمالي المنشورات والأزواج الممكنة، مع توزيع عدد التعليقات وأكبر المنشورات حجمًا للتحقق من كثافة البيانات قبل
    <span dir="ltr">pair mining</span>.
  </p>
</div>

In [ ]:
def post_pair_stats(df, post_col="post_id_true"):
    post_counts = (
        df.groupby(post_col)
        .size()
        .reset_index(name="comment_count")
    )

    post_counts["possible_pairs"] = (
        post_counts["comment_count"] * (post_counts["comment_count"] - 1) // 2
    )

    return post_counts


train_post_stats = post_pair_stats(train_df)
val_post_stats = post_pair_stats(val_df)

print("Train posts:", len(train_post_stats))
print("Train possible same-post pairs:", int(train_post_stats["possible_pairs"].sum()))

print("\nVal posts:", len(val_post_stats))
print("Val possible same-post pairs:", int(val_post_stats["possible_pairs"].sum()))

print("\nTrain comment count distribution:")
display(train_post_stats["comment_count"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nLargest train posts:")
display(train_post_stats.sort_values("comment_count", ascending=False).head(20))

print("\nVal comment count distribution:")
display(val_post_stats["comment_count"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

Train posts: 12023
Train possible same-post pairs: 22513411

Val posts: 2585
Val possible same-post pairs: 4709070

Train comment count distribution:


,comment_count
count,12023.000000
mean,49.367296
std,36.843010
min,1.000000
50%,41.000000
75%,90.000000
90%,100.000000
95%,100.000000
99%,100.000000
max,200.000000



Largest train posts:


,post_id_true,comment_count,possible_pairs
5483,10155613369808010,200,19900
5735,10155627531318010,200,19900
5570,10155617370993010,200,19900
5759,10155628646143010,198,19503
5796,10155631498273010,198,19503
5784,10155630749438010,198,19503
5805,10155632118313010,196,19110
5664,10155621858013010,194,18721
5859,10155635674288010,194,18721
5871,10155636469733010,194,18721



Val comment count distribution:


,comment_count
count,2585.000000
mean,48.593424
std,36.485183
min,1.000000
50%,40.000000
75%,88.000000
90%,100.000000
95%,100.000000
99%,100.000000
max,196.000000


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>12 — حفظ بيانات V7 الجاهزة لبناء الأزواج</h2>

  <p>
    تحفظ هذه الخلية ملفات
    <span dir="ltr">filtered_comments</span> و
    <span dir="ltr">train_df</span> و
    <span dir="ltr">val_df</span>
    داخل مجلد مخصص لمرحلة
    <span dir="ltr">V7 pair mining</span>.
  </p>

  <p>
    تنشئ أيضًا ملف
    <span dir="ltr">metadata.json</span>
    يحتوي على أعداد البيانات وإعدادات الفلترة، مع توضيح أن قسم
    <span dir="ltr">test</span>
    غير مستخدم في بناء أزواج التدريب.
  </p>
</div>

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

SAVE_DIR = PROJECT_ROOT / "training_data/v7_raw_prepared_for_pair_mining"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_COMMENTS_PATH = SAVE_DIR / "filtered_comments_v7_raw.csv"
TRAIN_DF_PATH = SAVE_DIR / "train_comments_v7_for_pair_mining.csv"
VAL_DF_PATH = SAVE_DIR / "val_comments_v7_for_pair_mining.csv"

filtered_comments.to_csv(FILTERED_COMMENTS_PATH, index=False)
train_df.to_csv(TRAIN_DF_PATH, index=False)
val_df.to_csv(VAL_DF_PATH, index=False)

metadata = {
    "filtered_comments": len(filtered_comments),
    "train_comments": len(train_df),
    "val_comments": len(val_df),
    "min_words_filter": MIN_WORDS_FOR_PAIR_MINING,
    "min_alpha_chars": MIN_ALPHA_CHARS,
    "note": "Prepared raw comments for V7 pair mining. Test split is not used for training pair mining."
}

METADATA_PATH = SAVE_DIR / "v7_raw_prepared_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved:")
print(FILTERED_COMMENTS_PATH)
print(TRAIN_DF_PATH)
print(VAL_DF_PATH)
print(METADATA_PATH)

Saved:
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_prepared_for_pair_mining/filtered_comments_v7_raw.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_prepared_for_pair_mining/train_comments_v7_for_pair_mining.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_prepared_for_pair_mining/val_comments_v7_for_pair_mining.csv
/content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_prepared_for_pair_mining/v7_raw_prepared_metadata.json


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>13 — تحميل بيانات V7 المحفوظة والتحقق منها</h2>

  <p>
    تعيد هذه الخلية تحميل ملفات
    <span dir="ltr">filtered_comments</span> و
    <span dir="ltr">train_df</span> و
    <span dir="ltr">val_df</span>
    من مجلد
    <span dir="ltr">V7 pair mining</span>.
  </p>

  <p>
    تقرأ ملف
    <span dir="ltr">metadata.json</span>
    ثم تعرض أبعاد الجداول وعينة من بيانات التدريب للتأكد من أن الحفظ والتحميل تما بنجاح.
  </p>
</div>

In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

SAVE_DIR = PROJECT_ROOT / "training_data/v7_raw_prepared_for_pair_mining"

FILTERED_COMMENTS_PATH = SAVE_DIR / "filtered_comments_v7_raw.csv"
TRAIN_DF_PATH = SAVE_DIR / "train_comments_v7_for_pair_mining.csv"
VAL_DF_PATH = SAVE_DIR / "val_comments_v7_for_pair_mining.csv"
METADATA_PATH = SAVE_DIR / "v7_raw_prepared_metadata.json"

filtered_comments = pd.read_csv(FILTERED_COMMENTS_PATH)
train_df = pd.read_csv(TRAIN_DF_PATH)
val_df = pd.read_csv(VAL_DF_PATH)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("filtered_comments:", filtered_comments.shape)
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print(metadata)

display(train_df.head())

/tmp/ipykernel_1372/2415018674.py:14: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  filtered_comments = pd.read_csv(FILTERED_COMMENTS_PATH)
/tmp/ipykernel_1372/2415018674.py:15: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_DF_PATH)


filtered_comments: (843636, 11)
train_df: (593543, 11)
val_df: (125614, 11)
{'filtered_comments': 843636, 'train_comments': 593543, 'val_comments': 125614, 'min_words_filter': 6, 'min_alpha_chars': 15, 'note': 'Prepared raw comments for V7 pair mining. Test split is not used for training pair mining.'}


,created_time,from_id,from_name,message,post_name,post_id_true,split,clean_text_v7,word_count,char_count,is_low_information
0,2017-07-14T14:43:54+0000,33661642d99eeceeb086,4ca212f16b9f954d5e0a,We are speaking to NRA supporters as well as W...,33661642d99eeceeb086_10154890879532217,10154890879532217,train,We are speaking to NRA supporters as well as W...,12,69,False
1,2017-07-14T14:41:59+0000,33661642d99eeceeb086,4ca212f16b9f954d5e0a,If you are just joining us we are outside of t...,33661642d99eeceeb086_10154890879532217,10154890879532217,train,If you are just joining us we are outside of t...,46,286,False
2,2017-07-14T14:41:58+0000,142b054b9d7f119260fa,210f51cc65568bf2d528,Do you know how backward America are in allowi...,142b054b9d7f119260fa_10154890879532217,10154890879532217,train,Do you know how backward America are in allowi...,29,163,False
3,2017-07-14T14:42:25+0000,52228acb5d5ca468be8c,46377b3cde64b2bf93dc,People who legally own guns often seem all too...,52228acb5d5ca468be8c_10154890879532217,10154890879532217,train,People who legally own guns often seem all too...,33,192,False
4,2017-07-14T14:40:46+0000,af7fe02906a110370810,087accbb0dc7c975d194,Have you snowflakes watched the news and seen ...,af7fe02906a110370810_10154890879532217,10154890879532217,train,Have you snowflakes watched the news and seen ...,23,124,False


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>14 — تحميل نموذج الـ Embeddings المستخدم في التعدين</h2>

  <p>
    تحمل هذه الخلية نموذج
    <span dir="ltr">paraphrase-multilingual-MiniLM-L12-v2</span>
    لاستخدامه في توليد التمثيلات الدلالية أثناء
    <span dir="ltr">pair mining</span>.
  </p>

  <p>
    تحدد الجهاز المستخدم
    <span dir="ltr">cuda</span>
    أو
    <span dir="ltr">cpu</span>
    وتضبط
    <span dir="ltr">max_seq_length = 256</span>
    ثم تعرض معلومات النموذج والـ
    <span dir="ltr">GPU</span>
    للتحقق.
  </p>
</div>

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MINING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

mining_model = SentenceTransformer(MINING_MODEL_NAME, device=device)
mining_model.max_seq_length = 256

print("Mining model:", MINING_MODEL_NAME)
print("Device:", device)
print("Max sequence length:", mining_model.max_seq_length)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Mining model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Device: cuda
Max sequence length: 256
GPU: NVIDIA A100-SXM4-40GB


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>15 — توليد وحفظ Embeddings للتدريب والتحقق</h2>

  <p>
    تحول هذه الخلية نصوص
    <span dir="ltr">train_df</span>
    و
    <span dir="ltr">val_df</span>
    إلى
    <span dir="ltr">normalized embeddings</span>
    باستخدام نموذج التعدين المحمّل سابقًا.
  </p>

  <p>
    تحفظ النتائج كملفات
    <span dir="ltr">.npy</span>
    لاستخدامها لاحقًا في مرحلة
    <span dir="ltr">pair mining</span>
    دون إعادة حساب التمثيلات من جديد.
  </p>
</div>

In [ ]:
TRAIN_TEXTS = train_df["clean_text_v7"].tolist()
VAL_TEXTS = val_df["clean_text_v7"].tolist()

print("Train texts:", len(TRAIN_TEXTS))
print("Val texts:", len(VAL_TEXTS))

train_embeddings = mining_model.encode(
    TRAIN_TEXTS,
    batch_size=1024,
    show_progress_bar=True,
    normalize_embeddings=True
)

val_embeddings = mining_model.encode(
    VAL_TEXTS,
    batch_size=1024,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Train embeddings:", train_embeddings.shape)
print("Val embeddings:", val_embeddings.shape)



np.save(TRAIN_EMB_OUT, train_embeddings)
np.save(VAL_EMB_OUT, val_embeddings)

print("Saved train embeddings:", TRAIN_EMB_OUT)
print("Saved val embeddings:", VAL_EMB_OUT)

Train texts: 593543
Val texts: 125614


Batches:   0%|          | 0/580 [00:00<?, ?it/s]

Batches:   0%|          | 0/123 [00:00<?, ?it/s]

Train embeddings: (593543, 384)
Val embeddings: (125614, 384)


NameError: name 'OUTPUT_DIR' is not defined

<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>16 — إنشاء مجلد تشغيل V7 وحفظ ملفات الـ Embeddings</h2>

  <p>
    تنشئ هذه الخلية مجلد إخراج جديد باسم
    <span dir="ltr">RUN_ID</span>
    لحفظ نتائج تشغيل
    <span dir="ltr">V7 raw pair mining</span>.
  </p>

  <p>
    تتحقق من وجود
    <span dir="ltr">train_embeddings</span>
    و
    <span dir="ltr">val_embeddings</span>
    ثم تحفظهما كملفات
    <span dir="ltr">.npy</span>
    داخل مجلد التشغيل.
  </p>
</div>

In [ ]:
from pathlib import Path
import time
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

RUN_ID = time.strftime("%Y%m%d_%H%M%S")

OUTPUT_DIR = PROJECT_ROOT / f"training_data/v7_raw_pair_mining_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("train_embeddings exists:", "train_embeddings" in globals())
print("val_embeddings exists:", "val_embeddings" in globals())

print("Train embeddings shape:", train_embeddings.shape)
print("Val embeddings shape:", val_embeddings.shape)

TRAIN_EMB_OUT = OUTPUT_DIR / "train_raw_v7_mining_embeddings.npy"
VAL_EMB_OUT = OUTPUT_DIR / "val_raw_v7_mining_embeddings.npy"

np.save(TRAIN_EMB_OUT, train_embeddings)
np.save(VAL_EMB_OUT, val_embeddings)

print("Saved train embeddings:", TRAIN_EMB_OUT)
print("Saved val embeddings:", VAL_EMB_OUT)

OUTPUT_DIR: /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_pair_mining_20260428_232654
train_embeddings exists: True
val_embeddings exists: True
Train embeddings shape: (593543, 384)
Val embeddings shape: (125614, 384)
Saved train embeddings: /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_pair_mining_20260428_232654/train_raw_v7_mining_embeddings.npy
Saved val embeddings: /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_pair_mining_20260428_232654/val_raw_v7_mining_embeddings.npy


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>17 — ضبط حدود وشروط بناء أزواج V7</h2>

  <p>
    تحدد هذه الخلية حدود التشابه الدلالي المقبولة بين التعليقات:
    <span dir="ltr">PAIR_MIN_SIM</span>
    و
    <span dir="ltr">PAIR_MAX_SIM</span>
    لاختيار أزواج مفيدة وغير متطابقة تقريبًا.
  </p>

  <p>
    تضبط أيضًا الحد الأقصى للتعليقات والأزواج لكل منشور، مع تحديد العدد النهائي لأزواج
    <span dir="ltr">train</span>
    و
    <span dir="ltr">val</span>
    قبل بدء مرحلة
    <span dir="ltr">pair mining</span>.
  </p>
</div>

In [ ]:
PAIR_MIN_SIM = 0.45
PAIR_MAX_SIM = 0.90

MAX_COMMENTS_PER_POST = 200
MAX_PAIRS_PER_POST_TRAIN = 5000
MAX_PAIRS_PER_POST_VAL = 1500

MAX_TRAIN_PAIRS_FINAL = 1_000_000
MAX_VAL_PAIRS_FINAL = 30_000

print("PAIR_MIN_SIM:", PAIR_MIN_SIM)
print("PAIR_MAX_SIM:", PAIR_MAX_SIM)
print("MAX_COMMENTS_PER_POST:", MAX_COMMENTS_PER_POST)
print("MAX_PAIRS_PER_POST_TRAIN:", MAX_PAIRS_PER_POST_TRAIN)
print("MAX_PAIRS_PER_POST_VAL:", MAX_PAIRS_PER_POST_VAL)
print("MAX_TRAIN_PAIRS_FINAL:", MAX_TRAIN_PAIRS_FINAL)
print("MAX_VAL_PAIRS_FINAL:", MAX_VAL_PAIRS_FINAL)

PAIR_MIN_SIM: 0.45
PAIR_MAX_SIM: 0.9
MAX_COMMENTS_PER_POST: 200
MAX_PAIRS_PER_POST_TRAIN: 5000
MAX_PAIRS_PER_POST_VAL: 1500
MAX_TRAIN_PAIRS_FINAL: 1000000
MAX_VAL_PAIRS_FINAL: 30000


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>18 — التحقق من جاهزية دالة التعدين والبيانات</h2>

  <p>
    تتحقق هذه الخلية من وجود الدالة
    <span dir="ltr">mine_pairs_from_same_post</span>
    ومن جاهزية
    <span dir="ltr">train_df</span> و
    <span dir="ltr">val_df</span>
    وملفات
    <span dir="ltr">embeddings</span>
    قبل تنفيذ مرحلة
    <span dir="ltr">pair mining</span>.
  </p>
</div>

In [ ]:
print("mine_pairs_from_same_post exists:", "mine_pairs_from_same_post" in globals())
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("train_embeddings:", train_embeddings.shape)
print("val_embeddings:", val_embeddings.shape)

mine_pairs_from_same_post exists: False
train_df: (593543, 11)
val_df: (125614, 11)
train_embeddings: (593543, 384)
val_embeddings: (125614, 384)


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>19 — تعريف دالة تعدين أزواج التعليقات داخل المنشور نفسه</h2>

  <p>
    تعرّف هذه الخلية الدالة
    <span dir="ltr">mine_pairs_from_same_post</span>
    لاستخراج أزواج تعليقات من المنشور نفسه اعتمادًا على تشابه
    <span dir="ltr">embeddings</span>.
  </p>

  <p>
    تختار الأزواج الواقعة بين
    <span dir="ltr">min_sim</span>
    و
    <span dir="ltr">max_sim</span>
    مع ضبط الحد الأقصى للتعليقات والأزواج لكل منشور لتجنب هيمنة المنشورات الكبيرة.
  </p>

  <p>
    تعيد الخلية
    <span dir="ltr">pairs_df</span>
    مع إحصاءات التعدين، وتحذف الأزواج المكررة قبل استخدامها في التدريب.
  </p>
</div>

In [ ]:
from sklearn.preprocessing import normalize
import numpy as np
import pandas as pd

def mine_pairs_from_same_post(
    df,
    embeddings,
    text_col="clean_text_v7",
    post_col="post_id_true",
    min_sim=0.45,
    max_sim=0.90,
    max_comments_per_post=200,
    max_pairs_per_post=5000,
    random_state=42,
    split_name="train",
):
    rng = np.random.default_rng(random_state)

    if len(df) != len(embeddings):
        raise ValueError(f"df and embeddings mismatch: {len(df)} vs {len(embeddings)}")

    emb = normalize(np.asarray(embeddings, dtype=np.float32), norm="l2")

    pairs = []
    stats = {
        "split": split_name,
        "posts_seen": 0,
        "posts_used": 0,
        "posts_skipped_lt2": 0,
        "candidate_pairs_after_threshold": 0,
        "saved_pairs_before_dedup": 0,
    }

    grouped = df.groupby(post_col).indices

    for post_id, idx_array in grouped.items():
        stats["posts_seen"] += 1

        idx_array = np.array(idx_array, dtype=int)
        n = len(idx_array)

        if n < 2:
            stats["posts_skipped_lt2"] += 1
            continue

        if max_comments_per_post is not None and n > max_comments_per_post:
            idx_array = rng.choice(idx_array, size=max_comments_per_post, replace=False)
            n = len(idx_array)

        local_emb = emb[idx_array]
        sim_matrix = local_emb @ local_emb.T

        upper_i, upper_j = np.triu_indices(n, k=1)
        sims = sim_matrix[upper_i, upper_j]

        mask = (sims >= min_sim) & (sims <= max_sim)

        if not np.any(mask):
            continue

        cand_i = upper_i[mask]
        cand_j = upper_j[mask]
        cand_sims = sims[mask]

        stats["candidate_pairs_after_threshold"] += len(cand_sims)

        if max_pairs_per_post is not None and len(cand_sims) > max_pairs_per_post:
            order = np.argsort(-cand_sims)[:max_pairs_per_post]
            cand_i = cand_i[order]
            cand_j = cand_j[order]
            cand_sims = cand_sims[order]

        for a_local, b_local, sim in zip(cand_i, cand_j, cand_sims):
            a_idx = idx_array[a_local]
            b_idx = idx_array[b_local]

            text_a = df.iloc[a_idx][text_col]
            text_b = df.iloc[b_idx][text_col]

            if text_a == text_b:
                continue

            pairs.append({
                "post_id_true": post_id,
                "text_a": text_a,
                "text_b": text_b,
                "similarity": float(sim),
                "split": split_name,
            })

        stats["posts_used"] += 1
        stats["saved_pairs_before_dedup"] = len(pairs)

        if stats["posts_seen"] % 500 == 0:
            print(
                f"[{split_name}] posts_seen={stats['posts_seen']:,}, "
                f"saved_pairs={len(pairs):,}, "
                f"candidate_pairs={stats['candidate_pairs_after_threshold']:,}"
            )

    pairs_df = pd.DataFrame(pairs)

    if len(pairs_df) > 0:
        pairs_df = pairs_df.drop_duplicates(subset=["text_a", "text_b"]).reset_index(drop=True)

    stats["saved_pairs_after_dedup"] = len(pairs_df)

    return pairs_df, stats

print("Function loaded:", "mine_pairs_from_same_post" in globals())

Function loaded: True


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>20 — تنفيذ تعدين أزواج التدريب من تعليقات المنشور نفسه</h2>

  <p>
    تشغّل هذه الخلية الدالة
    <span dir="ltr">mine_pairs_from_same_post</span>
    على بيانات
    <span dir="ltr">train_df</span>
    لاستخراج أزواج تدريب من تعليقات المنشور نفسه.
  </p>

  <p>
    تعتمد عملية الاختيار على حدود التشابه
    <span dir="ltr">PAIR_MIN_SIM</span>
    و
    <span dir="ltr">PAIR_MAX_SIM</span>
    مع تطبيق حدود الحجم لمنع سيطرة المنشورات الكبيرة.
  </p>

  <p>
    تعرض الخلية إحصاءات التعدين، وعدد الأزواج قبل الحد النهائي، مع عينة من الأزواج وتوزيع قيم
    <span dir="ltr">similarity</span>.
  </p>
</div>

In [ ]:
import json

train_pairs_raw_v7, train_pair_stats = mine_pairs_from_same_post(
    df=train_df,
    embeddings=train_embeddings,
    text_col="clean_text_v7",
    post_col="post_id_true",
    min_sim=PAIR_MIN_SIM,
    max_sim=PAIR_MAX_SIM,
    max_comments_per_post=MAX_COMMENTS_PER_POST,
    max_pairs_per_post=MAX_PAIRS_PER_POST_TRAIN,
    random_state=42,
    split_name="train",
)

print(json.dumps(train_pair_stats, indent=2))
print("Train pairs before final cap:", len(train_pairs_raw_v7))

display(train_pairs_raw_v7.head())
display(train_pairs_raw_v7["similarity"].describe())

[train] posts_seen=500, saved_pairs=92,526, candidate_pairs=92,526
[train] posts_seen=1,000, saved_pairs=260,471, candidate_pairs=260,471
[train] posts_seen=1,500, saved_pairs=374,843, candidate_pairs=374,843
[train] posts_seen=2,000, saved_pairs=458,674, candidate_pairs=458,674
[train] posts_seen=2,500, saved_pairs=540,176, candidate_pairs=540,176
[train] posts_seen=3,000, saved_pairs=729,389, candidate_pairs=729,389
[train] posts_seen=4,000, saved_pairs=994,925, candidate_pairs=994,925
[train] posts_seen=4,500, saved_pairs=1,296,484, candidate_pairs=1,296,484
[train] posts_seen=5,000, saved_pairs=1,465,662, candidate_pairs=1,465,662
[train] posts_seen=5,500, saved_pairs=1,618,600, candidate_pairs=1,618,600
[train] posts_seen=6,000, saved_pairs=1,768,363, candidate_pairs=1,768,363
[train] posts_seen=6,500, saved_pairs=1,874,668, candidate_pairs=1,874,668
[train] posts_seen=7,000, saved_pairs=1,949,185, candidate_pairs=1,949,185
[train] posts_seen=7,500, saved_pairs=2,060,180, candidat

,post_id_true,text_a,text_b,similarity,split
0,323319168113278,"Not condoning what the driver of the car did, ...",Those bikers do not pay road use taxes. Stay t...,0.655765,train
1,323319168113278,"Not condoning what the driver of the car did, ...",I'm just here to argue with people condoning w...,0.646439,train
2,323319168113278,"Not condoning what the driver of the car did, ...",Why was this rider in the middle of the road d...,0.575418,train
3,323319168113278,"Not condoning what the driver of the car did, ...",I think we can all agree the bicyclists was be...,0.714095,train
4,323319168113278,"Not condoning what the driver of the car did, ...",Get the bikes OFF THE ROADS that were paid for...,0.630809,train


,similarity
count,3.701173e+06
mean,5.381970e-01
std,7.062375e-02
min,4.500000e-01
25%,4.817538e-01
50%,5.214511e-01
75%,5.787594e-01
max,8.999673e-01


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>21 — فحص جودة الأزواج حسب نطاقات التشابه</h2>

  <p>
    تقسّم هذه الخلية أزواج التدريب إلى نطاقات تشابه منخفضة، متوسطة، وعالية لفحص جودة
    <span dir="ltr">same-post pairs</span>
    يدويًا.
  </p>

  <p>
    تعرض عينات من كل نطاق مع قيمة
    <span dir="ltr">similarity</span>
    و
    <span dir="ltr">post_id_true</span>
    للتأكد من أن حدود التشابه المختارة مناسبة قبل التدريب.
  </p>
</div>

In [ ]:
def sample_similarity_band(df, low, high, n=20, random_state=42):
    band = df[(df["similarity"] >= low) & (df["similarity"] < high)].copy()
    print(f"Band {low} - {high}: {len(band):,} pairs")

    if len(band) == 0:
        return pd.DataFrame()

    return band.sample(min(n, len(band)), random_state=random_state)[
        ["text_a", "text_b", "similarity", "post_id_true"]
    ]

low_band_sample = sample_similarity_band(train_pairs_raw_v7, 0.45, 0.55, n=20)
mid_band_sample = sample_similarity_band(train_pairs_raw_v7, 0.55, 0.75, n=20)
high_band_sample = sample_similarity_band(train_pairs_raw_v7, 0.75, 0.90, n=20)

print("LOW BAND SAMPLE")
display(low_band_sample)

print("MID BAND SAMPLE")
display(mid_band_sample)

print("HIGH BAND SAMPLE")
display(high_band_sample)

Band 0.45 - 0.55: 2,367,824 pairs
Band 0.55 - 0.75: 1,299,023 pairs
Band 0.75 - 0.9: 34,324 pairs
LOW BAND SAMPLE


,text_a,text_b,similarity,post_id_true
3550477,Their food is overpriced and made from crap. I...,Another way for McDonald's to lay off more wor...,0.508880,490394057971413
3292572,Megyn you need to move to CNN. We are getting ...,SELF PROMOTING FRAUD KELLY NOW TRIES TO BLOCK ...,0.454561,1775537272708737
1148811,Of course these bastards would come out with a...,Why? They wanted to make the House bill more m...,0.533336,10154717948859067
2273493,"Go back 5 years or so and ""craft"" simply wasn'...",Craft beer drinkers know. We know who makes ou...,0.509777,10155677207061323
394702,They ALL know that Medicare for all is the onl...,Stop trying to repeal the expansion of coverag...,0.526216,10155564970858675
3216966,If they want to waste more time and money inve...,Oh come on! Rs have gone to the Clinton bashin...,0.541707,1718718821472914
2578963,"That is not the way it should be, I am startin...","Excuse me! Comey is a failure, a total failure...",0.469294,10158938758395533
2722538,Now whose name is on the paperwork? Go back up...,"Geeze, I can't imagine why Trump Jr would meet...",0.472302,10159260545760389
815735,Shouldn't it count as conspiracy when every me...,With all the meddling the last administration ...,0.489162,10151228648504999
2463023,Media pretends they have massively incriminati...,Lol! I'm thinking Trump Sr may use his son as ...,0.476171,10156028176993812


MID BAND SAMPLE


,text_a,text_b,similarity,post_id_true
778393,The only chaos is caused by the liberal media....,Maybe the National Review should stick to real...,0.587806,10159106011450093
1365879,A terrible war crime and made worse by Japan's...,And the Japanese still continue to downplay or...,0.568090,10154875829732217
1561051,Soooo....Fox misinterprets the information...n...,So... The fake president is tweeting fake info...,0.629618,10155051916379087
3630227,This is how we justify a war with China and oh...,"OF course China won't give up their ""Trump"" ca...",0.604739,832383486921603
962252,A funeral isn't the place to further your poli...,Hey good for nothing mayor scumbag; Drop dead ...,0.619214,10154640683312541
778260,No he does not need intervention for his Tweet...,"Yeah, the author is correct....if he keeps twe...",0.615337,10159106011450093
1525202,Seriously...at what point in history do we say...,Lots of this in the current administration,0.602827,10154960875522144
2008977,The child deserves placement in a good foster ...,True. Any child abused this much is a tragedy....,0.571158,10155570108852235
2826321,From Russia with love. A new report in Bloombe...,He has never been secure about a thing in his ...,0.597956,1361790343905397
3149176,"Megyn, unfortunately you dumped on Trump right...",It's too bad there are a lot of haters respond...,0.574757,1655129261416206


HIGH BAND SAMPLE


,text_a,text_b,similarity,post_id_true
51626,"One day, I hope the real Neo Nazis show up to ...","well, liberals show us all how neo-Nazi's real...",0.761145,1523615691009099
3528052,Well done all! Thanks for caring about this an...,Great to see people helping animals why can't ...,0.752648,2538724582853941
3428823,"Seriously, they had to explain to us who the z...",Do liberals even know what zombies look like a...,0.762969,1891718037514884
1269125,why ? why would Putin want you ? bc stupid ppl...,"😂 ""why would he want me?"".... putin wants you ...",0.798619,10154835720245950
3045828,No. The only thing trump is going to ask Putin...,"OMG, of course not. Putin will play him like a...",0.804110,1637009949728584
1385908,Not really sure why it's so disgusting to have...,So why exactly is it disgusting/gross to have ...,0.763838,10154880120412217
2540352,She's just a racist big mouth ghetto. Bitch th...,If she looses the election she will cry out ra...,0.792559,10156596229514657
2070901,Having children is a selfish decision. I admit...,It's definitely more selfish to have children ...,0.819045,10155610624403487
748083,I love Waffle House! Just ate there last week....,We love Waffle House! From Florida to Tennesse...,0.768823,10157050314006509
2285916,I've reheated left over chilli to add to chees...,I think its just a taste preference being brit...,0.774385,10155681058571323


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>22 — حساب توزيع الأزواج حسب نطاقات التشابه</h2>

  <p>
    تقسم هذه الخلية أزواج
    <span dir="ltr">train_pairs_raw_v7</span>
    إلى ثلاث فئات تشابه:
    منخفض، متوسط، وعالٍ باستخدام
    <span dir="ltr">pd.cut</span>.
  </p>

  <p>
    تعرض عدد الأزواج داخل كل نطاق للتحقق من توازن بيانات
    <span dir="ltr">pair mining</span>
    قبل اعتمادها للتدريب.
  </p>
</div>

In [ ]:
pair_bins = pd.cut(
    train_pairs_raw_v7["similarity"],
    bins=[0.45, 0.55, 0.75, 0.90],
    right=False,
    labels=["low_045_055", "mid_055_075", "high_075_090"]
)

bin_counts = pair_bins.value_counts().sort_index()
display(bin_counts)

,count
similarity,
low_045_055,2367824
mid_055_075,1299023
high_075_090,34324


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>23 — اختيار عينة التدريب النهائية حسب نطاقات التشابه</h2>

  <p>
    تحدد هذه الخلية خطة أخذ عينات للوصول إلى
    <span dir="ltr">1,500,000</span>
    زوج تدريب موزعة على نطاقات تشابه منخفضة، متوسطة، وعالية.
  </p>

  <p>
    تستخدم الدالة
    <span dir="ltr">sample_by_similarity_plan</span>
    لاختيار العدد المطلوب من كل نطاق، ثم تخلط الأزواج عشوائيًا لإنتاج
    <span dir="ltr">train_pairs_v7</span>.
  </p>

  <p>
    تعرض الخلية العدد النهائي، إحصاءات
    <span dir="ltr">similarity</span>
    وتوزيع الأزواج حسب النطاقات للتحقق قبل التدريب.
  </p>
</div>

In [ ]:
TARGET_TRAIN_PAIRS = 1_500_000

TRAIN_BAND_PLAN = {
    "low_045_055": 166_653,
    "mid_055_075": 1_299_023,
    "high_075_090": 34_324,
}

BANDS = {
    "low_045_055": (0.45, 0.55),
    "mid_055_075": (0.55, 0.75),
    "high_075_090": (0.75, 0.90),
}

def sample_by_similarity_plan(df, band_plan, random_state=42):
    sampled_parts = []

    for band_name, target_n in band_plan.items():
        low, high = BANDS[band_name]

        band_df = df[
            (df["similarity"] >= low) &
            (df["similarity"] < high)
        ].copy()

        available = len(band_df)
        take_n = min(target_n, available)

        print(
            f"{band_name}: available={available:,}, "
            f"target={target_n:,}, taking={take_n:,}"
        )

        if take_n > 0:
            sampled_parts.append(
                band_df.sample(n=take_n, random_state=random_state)
            )

    final_df = pd.concat(sampled_parts, ignore_index=True)
    final_df = final_df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    return final_df


train_pairs_v7 = sample_by_similarity_plan(
    train_pairs_raw_v7,
    TRAIN_BAND_PLAN,
    random_state=42
)

print("Final train pairs:", len(train_pairs_v7))

print("\nFinal train similarity:")
display(train_pairs_v7["similarity"].describe())

print("\nFinal train band distribution:")
display(
    pd.cut(
        train_pairs_v7["similarity"],
        bins=[0.45, 0.55, 0.75, 0.90],
        right=False,
        labels=["low_045_055", "mid_055_075", "high_075_090"]
    ).value_counts().sort_index()
)

display(train_pairs_v7.sample(20, random_state=42))

low_045_055: available=2,367,824, target=166,653, taking=166,653
mid_055_075: available=1,299,023, target=1,299,023, taking=1,299,023
high_075_090: available=34,324, target=34,324, taking=34,324
Final train pairs: 1500000

Final train similarity:


,similarity
count,1.500000e+06
mean,6.024184e-01
std,6.472059e-02
min,4.500003e-01
25%,5.637642e-01
50%,5.940013e-01
75%,6.389512e-01
max,8.999673e-01



Final train band distribution:


,count
similarity,
low_045_055,166653
mid_055_075,1299023
high_075_090,34324


,post_id_true,text_a,text_b,similarity,split
610740,10154862794117217,Two of the world's most reviled leaders shakin...,OMG I THINK THEY JUST COLLUDED DURING THAT ONE...,0.463916,train
233172,10154871379042144,Let's not forget the doctors who over prescrib...,One very small town in Ohio had 9 million of t...,0.624530,train
1149767,10155608151473487,Is it more work to cut and style a woman's hai...,"I am a hairdresser, and I charge the same for ...",0.607726,train
446241,10155054157604087,congrats on proving the Trump campaign collude...,And to any remaining Trump supporters: if you ...,0.554387,train
31954,10155685359351323,I have little sympathy. She called this electi...,Tears for whom? The country for its lost direc...,0.561444,train
1246056,10155835114431756,A well-informed public not propagandized by th...,"I am a supporter of Medicare for all, however ...",0.550901,train
858262,10154755933562297,And the shame and embarrassment continues. I j...,"Let's see, if he can really pull it off! ""The ...",0.643884,train
935814,1559534910733526,I'll get behind the pledge before a government...,"SEPARATION OF CHURCH AND STATE, why is that so...",0.602584,train
151067,1620211148241351,Megan you and Doug look very happy !,"Beautiful couple in this photo, I love Megyn K...",0.560078,train
1309962,10155070537254160,Makes sense considering Trump has failed at es...,They'd not doing their job with trump either h...,0.592218,train


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>24 — تنفيذ تعدين أزواج التحقق من تعليقات المنشور نفسه</h2>

  <p>
    تشغّل هذه الخلية دالة
    <span dir="ltr">mine_pairs_from_same_post</span>
    على بيانات
    <span dir="ltr">val_df</span>
    لاستخراج أزواج
    <span dir="ltr">validation</span>
    ضمن حدود التشابه المحددة.
  </p>

  <p>
    تعرض إحصاءات التعدين، عدد الأزواج قبل الحد النهائي، توزيع
    <span dir="ltr">similarity</span>
    وتوزيع الأزواج حسب النطاقات للتحقق من جودة بيانات التقييم.
  </p>
</div>

In [ ]:
import json

val_pairs_raw_v7, val_pair_stats = mine_pairs_from_same_post(
    df=val_df,
    embeddings=val_embeddings,
    text_col="clean_text_v7",
    post_col="post_id_true",
    min_sim=PAIR_MIN_SIM,
    max_sim=PAIR_MAX_SIM,
    max_comments_per_post=MAX_COMMENTS_PER_POST,
    max_pairs_per_post=MAX_PAIRS_PER_POST_VAL,
    random_state=42,
    split_name="val",
)

print(json.dumps(val_pair_stats, indent=2))
print("Val pairs before final cap:", len(val_pairs_raw_v7))

display(val_pairs_raw_v7.head())
display(val_pairs_raw_v7["similarity"].describe())

print("\nVal band distribution:")
display(
    pd.cut(
        val_pairs_raw_v7["similarity"],
        bins=[0.45, 0.55, 0.75, 0.90],
        right=False,
        labels=["low_045_055", "mid_055_075", "high_075_090"]
    ).value_counts().sort_index()
)

[val] posts_seen=500, saved_pairs=112,022, candidate_pairs=121,775
[val] posts_seen=1,500, saved_pairs=469,639, candidate_pairs=502,327
[val] posts_seen=2,000, saved_pairs=555,742, candidate_pairs=597,944
[val] posts_seen=2,500, saved_pairs=726,581, candidate_pairs=773,961
{
  "split": "val",
  "posts_seen": 2585,
  "posts_used": 2377,
  "posts_skipped_lt2": 19,
  "candidate_pairs_after_threshold": 809682,
  "saved_pairs_before_dedup": 760846,
  "saved_pairs_after_dedup": 752789
}
Val pairs before final cap: 752789


,post_id_true,text_a,text_b,similarity,split
0,323510934760768,It's not cocky if you're backing it up! He's g...,He's an idiot and a right-wing extremist. Mich...,0.507467,val
1,323510934760768,It's not cocky if you're backing it up! He's g...,Anybody who criticizes this effort should just...,0.507814,val
2,323510934760768,It's not cocky if you're backing it up! He's g...,"So, the ""beer and condoms"" crowd of dullards w...",0.600641,val
3,323510934760768,It's not cocky if you're backing it up! He's g...,Kid rock smokes weed among other things im sur...,0.511211,val
4,323510934760768,It's not cocky if you're backing it up! He's g...,He's from Michigan? Must have been his neighbo...,0.492576,val


,similarity
count,752789.000000
mean,0.542582
std,0.071828
min,0.450000
25%,0.485074
50%,0.526631
75%,0.584531
max,0.899481



Val band distribution:


,count
similarity,
low_045_055,463078
mid_055_075,281596
high_075_090,8113


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>25 — اختيار عينة التحقق النهائية حسب نطاقات التشابه</h2>

  <p>
    تحدد هذه الخلية خطة أخذ عينات من أزواج
    <span dir="ltr">validation</span>
    موزعة على نطاقات التشابه المنخفضة، المتوسطة، والعالية.
  </p>

  <p>
    تنتج الخلية
    <span dir="ltr">val_pairs_v7</span>
    ثم تعرض العدد النهائي، إحصاءات
    <span dir="ltr">similarity</span>
    وتوزيع الأزواج حسب النطاقات للتحقق من توازن بيانات التقييم.
  </p>
</div>

In [ ]:
VAL_BAND_PLAN = {
    "low_045_055": 3_500,
    "mid_055_075": 25_000,
    "high_075_090": 1_500,
}

val_pairs_v7 = sample_by_similarity_plan(
    val_pairs_raw_v7,
    VAL_BAND_PLAN,
    random_state=42
)

print("Final val pairs:", len(val_pairs_v7))

print("\nFinal val similarity:")
display(val_pairs_v7["similarity"].describe())

print("\nFinal val band distribution:")
display(
    pd.cut(
        val_pairs_v7["similarity"],
        bins=[0.45, 0.55, 0.75, 0.90],
        right=False,
        labels=["low_045_055", "mid_055_075", "high_075_090"]
    ).value_counts().sort_index()
)

display(val_pairs_v7.sample(20, random_state=42))

low_045_055: available=463,078, target=3,500, taking=3,500
mid_055_075: available=281,596, target=25,000, taking=25,000
high_075_090: available=8,113, target=1,500, taking=1,500
Final val pairs: 30000

Final val similarity:


,similarity
count,30000.000000
mean,0.607473
std,0.071215
min,0.450041
25%,0.563961
50%,0.596498
75%,0.646003
max,0.899481



Final val band distribution:


,count
similarity,
low_045_055,3500
mid_055_075,25000
high_075_090,1500


,post_id_true,text_a,text_b,similarity,split
2308,10157047024121509,Lab rat Charlie. A moment of silence for this ...,This is disgusting to me. I feel for the paren...,0.451529,val
22404,1815219951826991,"Yeah, totally deferential. That's why he was a...",Hillary Clinton had already sold Putin 20% of ...,0.581408,val
23397,10154798072702297,No one is talking about banning guns except th...,It astonishing to see so many people crying ou...,0.697425,val
25058,10155770183334411,BREAKING NEWS ISRAEL KILL WOMEN AND CHILDREN E...,Now in retaliation Israel will rage war on Gaz...,0.572628,val
2664,10155628947109203,"All they had to say was ""History Channel"" and ...",The history channel is just propaganda and bul...,0.704540,val
8511,10156081919476037,If this was a guy he would be in jail. She nee...,She had better have to register as a sex offen...,0.560552,val
5148,1379756275416577,"Yet another ""diversion tactic""....make it look...",Nunes is not a coward.... I believe he will ge...,0.695956,val
7790,10159443360685354,McDonald's needs a security guard because of a...,Since when does McDonald's have Security Guard...,0.703826,val
11311,1655236228072176,"Fox- please support Trump. If you don't, we mi...",Not a good night for Fox. No Trump interview f...,0.559123,val
19043,1519427251427943,She is the most disrespectful and dishonest hu...,Total idiot. She is now the darling of the Dem...,0.581744,val


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>26 — حفظ أزواج V7 النهائية وملف التوثيق</h2>

  <p>
    تحفظ هذه الخلية أزواج التدريب
    <span dir="ltr">train_pairs_v7</span>
    وأزواج التحقق
    <span dir="ltr">val_pairs_v7</span>
    كملفات
    <span dir="ltr">CSV</span>.
  </p>

  <p>
    تنشئ أيضًا ملف
    <span dir="ltr">metadata.json</span>
    يوثق إعدادات التعدين، حدود التشابه، خطة أخذ العينات، أعداد الأزواج، وملخصات
    <span dir="ltr">similarity</span>
    لضمان قابلية التتبع وإعادة الإنتاج.
  </p>
</div>

In [ ]:
import json

TRAIN_PAIRS_OUT = OUTPUT_DIR / "train_stage1_pairs_v7_raw_large_balanced_1p5M.csv"
VAL_PAIRS_OUT = OUTPUT_DIR / "val_stage1_pairs_v7_raw_large_balanced_30k.csv"

train_pairs_v7.to_csv(TRAIN_PAIRS_OUT, index=False)
val_pairs_v7.to_csv(VAL_PAIRS_OUT, index=False)

metadata = {
    "run_id": OUTPUT_DIR.name,
    "source": "raw_comments",
    "output_dir": str(OUTPUT_DIR),

    "train_comments_for_pair_mining": int(len(train_df)),
    "val_comments_for_pair_mining": int(len(val_df)),

    "train_embeddings_path": str(OUTPUT_DIR / "train_raw_v7_mining_embeddings.npy"),
    "val_embeddings_path": str(OUTPUT_DIR / "val_raw_v7_mining_embeddings.npy"),

    "pair_min_sim": float(PAIR_MIN_SIM),
    "pair_max_sim": float(PAIR_MAX_SIM),

    "max_comments_per_post": int(MAX_COMMENTS_PER_POST),
    "max_pairs_per_post_train": int(MAX_PAIRS_PER_POST_TRAIN),
    "max_pairs_per_post_val": int(MAX_PAIRS_PER_POST_VAL),

    "train_pairs_before_final_sampling": int(len(train_pairs_raw_v7)),
    "val_pairs_before_final_sampling": int(len(val_pairs_raw_v7)),

    "train_sampling_strategy": "balanced_by_similarity_bands_1p5M",
    "val_sampling_strategy": "balanced_by_similarity_bands_30k",

    "train_band_plan": TRAIN_BAND_PLAN,
    "val_band_plan": VAL_BAND_PLAN,

    "final_train_pairs": int(len(train_pairs_v7)),
    "final_val_pairs": int(len(val_pairs_v7)),

    "train_similarity_summary": train_pairs_v7["similarity"].describe().to_dict(),
    "val_similarity_summary": val_pairs_v7["similarity"].describe().to_dict(),

    "train_pair_stats": train_pair_stats,
    "val_pair_stats": val_pair_stats,

    "random_state": 42,
}

METADATA_OUT = OUTPUT_DIR / "v7_raw_large_balanced_pair_mining_metadata.json"

with open(METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved train pairs:", TRAIN_PAIRS_OUT)
print("Saved val pairs:", VAL_PAIRS_OUT)
print("Saved metadata:", METADATA_OUT)

print("\nTrain final similarity:")
display(train_pairs_v7["similarity"].describe())

print("\nVal final similarity:")
display(val_pairs_v7["similarity"].describe())

Saved train pairs: /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_pair_mining_20260428_232654/train_stage1_pairs_v7_raw_large_balanced_1p5M.csv
Saved val pairs: /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_pair_mining_20260428_232654/val_stage1_pairs_v7_raw_large_balanced_30k.csv
Saved metadata: /content/drive/MyDrive/semester project/news_comment_topic_system/training_data/v7_raw_pair_mining_20260428_232654/v7_raw_large_balanced_pair_mining_metadata.json

Train final similarity:


,similarity
count,1.500000e+06
mean,6.024184e-01
std,6.472059e-02
min,4.500003e-01
25%,5.637642e-01
50%,5.940013e-01
75%,6.389512e-01
max,8.999673e-01



Val final similarity:


,similarity
count,30000.000000
mean,0.607473
std,0.071215
min,0.450041
25%,0.563961
50%,0.596498
75%,0.646003
max,0.899481
